# 03 交互式 Demo -- ipywidgets 聊天界面

完整体验教学闭环：苏格拉底式对话 + 实时出题 + 评估 + 错题整理。

| Cell | 内容 |
|---|---|
| 1 | 初始化：创建会话、checkpointer、Agent |
| 2 | UI 组件：对话区 + 答题卡 + 输入区 + 工具日志 |
| 3 | 交互逻辑：发送/答题/跳过/结束 + 事件提取 |
| 4 | 显示界面 |

**使用流程：**
1. 依次运行 Cell 1-4
2. 在输入框输入问题，点「发送」
3. 点「考考我」触发出题，答题卡弹出
4. 选择答案后点「提交答案」
5. 点「结束会话」整理错题

In [ ]:
# Cell 1: 初始化 -- 创建会话、checkpointer、Agent
import asyncio
import uuid
from datetime import datetime, timezone

import ipywidgets as widgets
from IPython.display import display, HTML
from sqlmodel import select

from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.types import Command

from teaching_agent.config import settings
from teaching_agent.db import async_session_factory
from teaching_agent.models import Session as SessionModel, Message, StudentProfile
from teaching_agent.agents.free_agent import create_free_agent

# 生成唯一会话 ID
session_id = f"demo-{uuid.uuid4().hex[:8]}"

# 创建会话记录 + 确保画像存在
async with async_session_factory() as session:
    session.add(SessionModel(
        id=session_id, user_id="demo-student", type="free",
        status="active", started_at=datetime.now(timezone.utc),
    ))
    # 确保学生画像存在
    result = await session.execute(
        select(StudentProfile).where(StudentProfile.user_id == "demo-student")
    )
    if not result.scalar_one_or_none():
        session.add(StudentProfile(
            id=str(uuid.uuid4()),
            user_id="demo-student",
            summary="新学生",
            weak_points=[],
            learning_style="未知",
        ))
    await session.commit()
print(f"✓ 会话创建: {session_id}")

# 创建 checkpointer（用 __aenter__ 保持连接开启，跨 cell 复用）
db_url = settings.DATABASE_URL.replace("+asyncpg", "")
_cm = AsyncPostgresSaver.from_conn_string(db_url)
checkpointer = await _cm.__aenter__()
await checkpointer.setup()
print("✓ Checkpointer 就绪")

# 创建 Agent
agent = await create_free_agent(checkpointer, session_id)
config = {
    "configurable": {"thread_id": session_id},
    "recursion_limit": settings.RECURSION_LIMIT,
}
print(f"✓ Agent 就绪，课程: {settings.COURSE_NAME}")

In [ ]:
# Cell 2: UI 组件
# 对话区域（滚动）
chat_output = widgets.Output(layout=widgets.Layout(
    height="400px", overflow_y="auto",
    border="1px solid #ddd", padding="10px",
))

# 答题卡区域（出题时显示）
quiz_stem_label = widgets.HTML()
quiz_answer_area = widgets.VBox([])
quiz_submit_btn = widgets.Button(description="提交答案", button_style="primary")
quiz_skip_btn = widgets.Button(description="跳过", button_style="warning")
quiz_box = widgets.VBox(
    [quiz_stem_label, quiz_answer_area,
     widgets.HBox([quiz_submit_btn, quiz_skip_btn])],
    layout=widgets.Layout(
        display="none", border="2px solid #2196f3",
        padding="10px", margin="5px 0",
    ),
)

# 输入区域
input_box = widgets.Text(placeholder="输入消息...", layout=widgets.Layout(width="50%"))
send_btn = widgets.Button(description="发送", button_style="primary", icon="paper-plane")
quiz_btn = widgets.Button(description="考考我", button_style="info")
end_btn = widgets.Button(description="结束会话", button_style="danger")
input_area = widgets.HBox([input_box, send_btn, quiz_btn, end_btn])

# 侧边栏：工具日志
log_output = widgets.Output(layout=widgets.Layout(
    height="435px", overflow_y="auto",
    border="1px solid #ddd", padding="10px",
))

# 整体布局
ui = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3>📚 教学助手</h3>"),
        chat_output,
        quiz_box,
        input_area,
    ], layout=widgets.Layout(width="70%")),
    widgets.VBox([
        widgets.HTML("<h3>📋 工具日志</h3>"),
        log_output,
    ], layout=widgets.Layout(width="30%")),
])
print("✓ UI 组件创建完成")

In [ ]:
# Cell 3: 交互逻辑
current_answer_widget = None


def log(msg: str):
    """记录到侧边栏。"""
    t = datetime.now().strftime("%H:%M:%S")
    with log_output:
        print(f"[{t}] {msg}")


def render_message(role: str, content: str):
    """在对话区域渲染消息气泡。"""
    if role == "user":
        html = (
            f'<div style="text-align:right;margin:8px 0;">'
            f'<span style="background:#e3f2fd;padding:8px 12px;border-radius:12px;'
            f'display:inline-block;max-width:80%;">'
            f'<b>学生</b><br>{content}</span></div>'
        )
    else:
        html = (
            f'<div style="text-align:left;margin:8px 0;">'
            f'<span style="background:#f5f5f5;padding:8px 12px;border-radius:12px;'
            f'display:inline-block;max-width:80%;">'
            f'<b>老师</b><br>{content}</span></div>'
        )
    with chat_output:
        display(HTML(html))


async def write_message(role: str, content: str):
    """写入 Message 表（业务记录）。"""
    async with async_session_factory() as session:
        session.add(Message(
            id=str(uuid.uuid4()),
            session_id=session_id,
            role=role,
            content=content,
        ))
        await session.commit()


def set_busy(busy: bool):
    """禁用/启用输入控件。"""
    input_box.disabled = busy
    send_btn.disabled = busy
    quiz_btn.disabled = busy
    end_btn.disabled = busy


def show_quiz(quiz_data: dict):
    """显示答题卡。"""
    global current_answer_widget
    quiz_stem_label.value = f"<b>📝 {quiz_data['stem']}</b>"
    q_type = quiz_data.get("q_type", "short")
    options = quiz_data.get("options", [])

    if q_type == "single" and options:
        current_answer_widget = widgets.RadioButtons(options=options)
    elif q_type == "multi" and options:
        current_answer_widget = widgets.SelectMultiple(options=options)
    else:
        current_answer_widget = widgets.Text(placeholder="输入你的答案...")

    quiz_answer_area.children = [current_answer_widget]
    quiz_box.layout.display = "flex"
    log(f"📝 出题 ({q_type}): {quiz_data['stem'][:60]}...")


def hide_quiz():
    """隐藏答题卡。"""
    quiz_box.layout.display = "none"


def get_answer() -> str:
    """从当前答题控件获取答案文本。"""
    if current_answer_widget is None:
        return ""
    if isinstance(current_answer_widget, widgets.RadioButtons):
        return current_answer_widget.value
    elif isinstance(current_answer_widget, widgets.SelectMultiple):
        return ", ".join(current_answer_widget.value)
    elif isinstance(current_answer_widget, widgets.Text):
        return current_answer_widget.value
    return ""


async def send_message(content: str):
    """发送消息并处理 Agent 响应。"""
    set_busy(True)
    render_message("user", content)
    await write_message("user", content)
    log(f"📤 发送: {content}")

    try:
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": content}]},
            config=config,
        )
        state = await agent.aget_state(config)

        if state.next:
            # Agent 被 interrupt（出题了）
            quiz_data = state.interrupts[0].value
            show_quiz(quiz_data)
        else:
            # Agent 完成回复
            reply = result["messages"][-1].content
            render_message("agent", reply)
            await write_message("agent", reply)
            log(f"📥 回复: {reply[:60]}...")
    except Exception as e:
        render_message("agent", f"⚠️ 发生错误: {e}")
        log(f"❌ 错误: {e}")
    finally:
        set_busy(False)


async def submit_answer(answer: str = "", skip: bool = False):
    """提交答案，从 interrupt 恢复。"""
    set_busy(True)
    hide_quiz()

    if skip:
        render_message("user", "⏭️ 跳过本题")
        log("⏭️ 跳过")
        resume_value = {"action": "skip"}
    else:
        render_message("user", f"答案: {answer}")
        log(f"✏️ 提交: {answer}")
        resume_value = {"answer": answer}

    try:
        result = await agent.ainvoke(
            Command(resume=resume_value),
            config=config,
        )
        state = await agent.aget_state(config)

        if state.next:
            # 可能又出题了
            quiz_data = state.interrupts[0].value
            show_quiz(quiz_data)
        else:
            reply = result["messages"][-1].content
            render_message("agent", reply)
            await write_message("agent", reply)
            log(f"📥 回复: {reply[:60]}...")
    except Exception as e:
        render_message("agent", f"⚠️ 发生错误: {e}")
        log(f"❌ 错误: {e}")
    finally:
        set_busy(False)


# === 按钮事件绑定 ===
def on_send_click(b):
    content = input_box.value.strip()
    if content:
        input_box.value = ""
        asyncio.create_task(send_message(content))

def on_quiz_trigger(b):
    asyncio.create_task(send_message("考考我"))

def on_end_click(b):
    asyncio.create_task(send_message("结束会话"))

def on_quiz_submit(b):
    answer = get_answer()
    if answer:
        asyncio.create_task(submit_answer(answer))

def on_quiz_skip(b):
    asyncio.create_task(submit_answer(skip=True))

send_btn.on_click(on_send_click)
quiz_btn.on_click(on_quiz_trigger)
end_btn.on_click(on_end_click)
quiz_submit_btn.on_click(on_quiz_submit)
quiz_skip_btn.on_click(on_quiz_skip)

# Enter 键发送（ipywidgets 8.x 兼容）
try:
    input_box.on_submit(lambda _: on_send_click(None))
except Exception:
    pass

print("✓ 交互逻辑就绪")

In [ ]:
# Cell 4: 显示界面
display(ui)
render_message("agent", f"你好！我是你的《{settings.COURSE_NAME}》学习助手。\n\n可以问我任何问题，或者点击「考考我」来测验。点击「结束会话」整理错题。")
log("✅ Demo 就绪")